# Handling Imbalanced Datasets ⚖️

This notebook shows three common techniques for handling imbalanced classification data:

- Random Oversampling
- Random Undersampling
- SMOTE

We use the Breast Cancer dataset and create an imbalanced version for practice.


In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Tool to create an imbalanced version of the dataset
from imblearn.datasets import make_imbalance

# Oversampling methods
from imblearn.over_sampling import SMOTE, RandomOverSampler

# Undersampling method
from imblearn.under_sampling import RandomUnderSampler

In [2]:
# Load the Breast Cancer dataset as a pandas DataFrame
data = load_breast_cancer(as_frame=True)  # load as dataframe

# Split data into input features (X) and target label (y)
X = data.data
y = data.target

# Combine features and target for quick inspection
df = X.copy()
df["target"] = y
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [3]:
# Check the original class distribution
df["target"].value_counts()

target
1    357
0    212
Name: count, dtype: int64

In [4]:
# Check whether any categorical columns are present
# SMOTE works directly with numerical features, so this check is important.
if len(df.select_dtypes(include=["object", "category"]).columns) > 0:
    print("Categorical columns exist.")
else:
    print("No categorical columns found.")

# Conclusion: all features are numerical, so SMOTE can be used safely here.

No categorical columns found.


## 📌 1. Key Idea: Resampling Methods


When a dataset is imbalanced, one class has many more rows than another class.

Resampling helps balance the target classes before model training:

- **Random Oversampling:** duplicates rows from the minority class.
- **Random Undersampling:** removes rows from the majority class.
- **SMOTE:** creates new synthetic minority-class rows using nearest neighbors.


In [5]:
# Create a strongly imbalanced dataset for demonstration
X_imbalance, y_imbalance = make_imbalance(
    X, y, sampling_strategy={0: 50, 1: 350}, random_state=42
)
y_imbalance.value_counts()

target
1    350
0     50
Name: count, dtype: int64

In [6]:
# Split the imbalanced data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_imbalance,
    y_imbalance,
    test_size=0.2,
    stratify=y_imbalance,  # keeps class percentages similar in train and test data
    random_state=42,
)
print("Training class distribution")
print(y_train.value_counts())
print("Testing class distribution")
print(y_test.value_counts())

# Because stratify is used, both train and test sets keep a similar 0/1 class ratio.

Training class distribution
target
1    280
0     40
Name: count, dtype: int64
Testing class distribution
target
1    70
0    10
Name: count, dtype: int64


## 🔁 2. Random Oversampling

Random oversampling balances the classes by duplicating rows from the minority class.


In [7]:
# Create the random oversampler
random_over_sampler = RandomOverSampler(
    sampling_strategy="auto",
    random_state=42,
)

# Apply oversampling only on the training data
X_test_over, y_test_over = random_over_sampler.fit_resample(X_train, y_train)

print("Before Oversampling")
print(X_train.shape, y_train.shape)
print("After Oversampling")
print(X_test_over.shape, y_test_over.shape)

Before Oversampling
(320, 30) (320,)
After Oversampling
(560, 30) (560,)


In [8]:
# Random oversampling duplicates existing minority-class rows
duplicated = X_test_over.duplicated()
duplicated.value_counts()  # True means the row is duplicated

False    320
True     240
Name: count, dtype: int64

## ✂️ 3. Random Undersampling

Random undersampling balances the classes by removing rows from the majority class.


In [9]:
# Create and apply the random undersampler
random_under_sampler = RandomUnderSampler(sampling_strategy="auto", random_state=42)
X_train_under, y_train_under = random_under_sampler.fit_resample(X_train, y_train)

print("Before Undersampling")
print(X_train.shape, y_train.shape)
print("y_train value counts", y_train.value_counts())
print("After Undersampling")
print(X_train_under.shape, y_train_under.shape)
print("y_train_under value counts", y_train_under.value_counts())

# In undersampling, majority-class rows are randomly removed to match the minority class.

Before Undersampling
(320, 30) (320,)
y_train value counts target
1    280
0     40
Name: count, dtype: int64
After Undersampling
(80, 30) (80,)
y_train_under value counts target
0    40
1    40
Name: count, dtype: int64


## 🧬 4. SMOTE

SMOTE balances the data by creating new synthetic minority-class rows instead of simply duplicating existing rows.


In [10]:
# Create and apply SMOTE on the training data
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(y_train_smote.value_counts())

# SMOTE does not duplicate rows directly.
# It creates new points between existing minority-class samples and their nearest neighbors.

target
1    280
0    280
Name: count, dtype: int64


## ✅ Quick Summary

- **Random Oversampling** increases the minority class by duplicating existing rows.
- **Random Undersampling** reduces the majority class by removing rows.
- **SMOTE** creates new synthetic rows for the minority class.
- Resampling should be applied on the training data only, not before the train-test split.
